In [ ]:
import numpy as np
from matplotlib.collections import LineCollection

# Les deux sens d'une meme liaison sont ramenes a une seule paire.
edge_pairs = data.edge_index.detach().cpu().numpy().T
edge_pairs = np.sort(edge_pairs, axis=1)
edge_pairs = np.unique(edge_pairs, axis=0)
edge_pairs = edge_pairs[edge_pairs[:, 0] != edge_pairs[:, 1]]

coordinates = data.x[:, :2].detach().cpu().numpy()
longitudes = coordinates[:, 0]
latitudes = coordinates[:, 1]

# Les liaisons traversant l'antimeridien sont masquees pour eviter une ligne sur toute la carte.
route_segments = []
for source, target in edge_pairs:
    if abs(longitudes[source] - longitudes[target]) <= 180:
        route_segments.append([
            (longitudes[source], latitudes[source]),
            (longitudes[target], latitudes[target]),
        ])

connection_counts = np.bincount(edge_pairs.ravel(), minlength=data.num_nodes)

fig, ax = plt.subplots(figsize=(17, 8), facecolor="#f4f7f6")
ax.set_facecolor("#e8f0f0")
ax.add_collection(LineCollection(
    route_segments,
    colors="#287c83",
    linewidths=0.35,
    alpha=0.18,
    zorder=1,
))

points = ax.scatter(
    longitudes,
    latitudes,
    c=np.log1p(connection_counts),
    s=8 + 28 * np.log1p(connection_counts) / np.log1p(connection_counts).max(),
    cmap="YlOrRd",
    edgecolors="white",
    linewidths=0.25,
    alpha=0.9,
    zorder=2,
)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_xticks(np.arange(-180, 181, 60))
ax.set_yticks(np.arange(-60, 86, 30))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(color="white", linewidth=0.8, alpha=0.9)
ax.set_title(
    f"Réseau mondial des aéroports · {data.num_nodes:,} aéroports · {len(edge_pairs):,} liaisons",
    loc="left",
    fontsize=15,
    pad=14,
)
colorbar = fig.colorbar(points, ax=ax, pad=0.02, shrink=0.78)
colorbar.set_label("Nombre de liaisons (échelle logarithmique)")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from matplotlib.collections import LineCollection

seuil_prediction = 0.5
model_c.eval()
model_device = next(model_c.parameters()).device
with th.no_grad():
    z_test = model_c.encode(
        test_data.x.to(model_device),
        test_data.edge_index.to(model_device),
    )
    edges_test = th.cat([
        test_data.pos_edge_label_index,
        test_data.neg_edge_label_index,
    ], dim=1).to(model_device)
    scores_test = model_c.decode(z_test, edges_test, sigmoid=True).cpu()

nb_aretes_positives = test_data.pos_edge_label_index.size(1)
scores_positives = scores_test[:nb_aretes_positives]
scores_negatives = scores_test[nb_aretes_positives:]

# Les liens positifs sous le seuil sont des faux négatifs.
liens_non_predits = test_data.pos_edge_label_index[:, scores_positives < seuil_prediction]
# Les liens négatifs au-dessus du seuil sont des faux positifs.
liens_predits_a_tort = test_data.neg_edge_label_index[:, scores_negatives >= seuil_prediction]

coords_carte = data.x[:, :2].detach().cpu().numpy()

def creer_segments_uniques(edge_index):
    paires = edge_index.detach().cpu().numpy().T
    paires = np.sort(paires, axis=1)
    paires = np.unique(paires, axis=0)
    paires = paires[paires[:, 0] != paires[:, 1]]
    segments = [
        [coords_carte[source], coords_carte[target]]
        for source, target in paires
        if abs(coords_carte[source, 0] - coords_carte[target, 0]) <= 180
    ]
    return segments, len(paires)

cartes_liens = [
    (data.edge_index, "Liaisons réelles", "#287c83"),
    (liens_non_predits, "Liaisons réelles non prédites (faux négatifs)", "#d99a25"),
    (liens_predits_a_tort, "Liaisons prédites à tort (faux positifs)", "#d65a3a"),
]

fig, axes = plt.subplots(3, 1, figsize=(16, 18), facecolor="#f4f7f6")
for ax, (aretes, titre, couleur) in zip(axes, cartes_liens):
    segments, nombre_liens = creer_segments_uniques(aretes)
    ax.set_facecolor("#e8f0f0")
    ax.add_collection(LineCollection(
        segments,
        colors=couleur,
        linewidths=0.45,
        alpha=0.45,
        zorder=1,
    ))
    ax.scatter(
        coords_carte[:, 0], coords_carte[:, 1],
        s=2, color="#18343a", alpha=0.45, zorder=2,
    )
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_xticks(np.arange(-180, 181, 60))
    ax.set_yticks(np.arange(-60, 86, 30))
    ax.grid(color="white", linewidth=0.8, alpha=0.9)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{titre} · {nombre_liens:,} liaisons", loc="left", fontsize=13, pad=10)
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle(
    f"Réseau des aéroports et erreurs de model_c (seuil = {seuil_prediction:.1f})",
    x=0.08,
    ha="left",
    fontsize=17,
)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

print(f"Faux négatifs sur le jeu de test : {liens_non_predits.size(1):,}")
print(f"Faux positifs sur le jeu de test : {liens_predits_a_tort.size(1):,}")